In [ ]:
# Important Imports, System Parameters and Drive Parameters:
import dynamiqs as dq
import jax.numpy as jnp
from pathlib import Path
import json
import matplotlib.pyplot as plt 
import jax

dq.set_device

dq.set_precision('double')
print(jax.devices())
# System Parameters:
params = {
    "W_c": 3030.0, 
    "Xcc" : 164.0, 
    "Kappa_c" : 0.0207,   

    "W_a": 4620.0, 
    "Xaa" : 0.0014, 
    "Kappa_a" : 0.0,

    "W_b": 7950.0, 
    "Xbb" : 0.0373, 
    "Kappa_b" : 0.419,

    "Xab" : 0.0,
    "Xac" : 0.042,
    "Xbc" : 1.3
}

# Drive Parameters:
params["Delta_c"] = 0.0
params["Delta_a"] = 20.0
params["Delta_b"] = 20.0
params["drive_amplitude_c"] = 20.0
params["drive_amplitude_a"] = 20.0
params["drive_amplitude_b"] = 0.6525

# Computing steady states : 
SS_a = abs(2*jnp.pi*params["drive_amplitude_a"]/(1j*2*jnp.pi*params["Delta_a"]-0.5*params["Kappa_a"]))**2
SS_b = abs(2*jnp.pi*params["drive_amplitude_b"]/(1j*2*jnp.pi*params["Delta_b"]-0.5*params["Kappa_b"]))**2

# Updating Drive Amplitude Conditions :
params["Delta_c"] = -SS_a*params["Xac"] - SS_b*params["Xbc"]
gca = jnp.sqrt(SS_a)*params["Xac"]
gcb = jnp.sqrt(SS_b)*params["Xbc"]

print(gca)
print(gcb)
print(SS_a)
print(SS_b)


# Defining the System : 
Na = 5
Nb = 5
Nc = 3
NN = Na*Nb*Nc
a = dq.destroy(Na); a = dq.tensor(a, dq.eye(Nb), dq.eye(Nc))
b = dq.destroy(Nb); b = dq.tensor(dq.eye(Na), b, dq.eye(Nc))
c = dq.destroy(Nc); c = dq.tensor(dq.eye(Na), dq.eye(Nb), c)

Numc = c.dag() @ c
Numa = a.dag() @ a
Numb = b.dag() @ b
I = dq.tensor(dq.eye(Na), dq.eye(Nb), dq.eye(Nc))

print(Numa.shape)
print(Numb.shape)
print(Numc.shape)
print(I.shape)

# Defining the Hamiltonian, Dissipator and Observables :
H0 = -2*jnp.pi * (params["Delta_c"]*Numc + 0.5*params["Xcc"]*Numc @ (Numc-I) + params["Delta_b"]*Numb + 0.5*params["Xbb"]*Numb @ (Numb-I))
H0 = H0 -2*jnp.pi * (params["Delta_a"]*Numa + 0.5*params["Xaa"]*Numa @ (Numa-I) + params["Xab"]*Numa@Numb+ params["Xac"]*Numa@Numc+ params["Xbc"]*Numb@Numc)                                                                                                                     



Hdrive = -2*jnp.pi * (0.5*params["drive_amplitude_c"]*(c+c.dag()) + params["drive_amplitude_a"]*(a+a.dag()) + params["drive_amplitude_b"]*(b+b.dag()))

H = H0 + Hdrive
Dissipators = [jnp.sqrt(params["Kappa_b"])*b]
Observables = [Numa, Numb, Numc]

# Defining the Scope of Simulation : 
times = jnp.linspace(0, 100, 1000)

# Initialization :
# 1. Calculate the actual coherent state amplitudes (alpha = -drive / detuning)
# (They are negative due to the sign convention in your Hamiltonian)
alpha_a = -params["drive_amplitude_a"] / params["Delta_a"] 
alpha_b = -params["drive_amplitude_b"] / (params["Delta_b"] - 1j * params["Kappa_b"] / (4*jnp.pi))

# 2. Create the coherent state density matrices
# Note: depending on your dynamiqs version, you may need to construct the dm from kets
dm_a = dq.coherent_dm(Na, alpha_a)
dm_b = dq.coherent_dm(Nb, alpha_b)


# 3. Initialize the system in the displaced state, with the transmon in |1>
# The transmon should start in |1> (fock state 1) to see the reset down to |0>
Rho0 = dq.tensor(dm_a, dm_b, dq.fock_dm(Nc, 0))


Method = dq.method.Tsit5(rtol = 1e-6, atol = 1e-8, max_steps = 5000000)
options = dq.options.Options(save_states=False)
#Method = dq.method.Expm()
# Running the Simulation :
result = dq.mesolve(H, Dissipators, Rho0, tsave=times, exp_ops=Observables, method=Method,options=options)  
print(result)
# Analyzing the Results :
#states_array = [rho.to_jax() for rho in result.states]
#Populattion_cavity = jnp.array([jnp.diag(rho).real for rho in states_array])

fig, ax = plt.subplots(figsize=(16,4))
ax.plot(times, result.expects[0].real, linewidth=2, color = "r")
ax.plot(times, result.expects[1].real, linewidth=2, color = "b")
ax.plot(times, result.expects[2].real, linewidth=2, color = "k")
ax.set_xlabel("Time (us)")
ax.set_ylabel("Photon Number")
ax.set_title("Transmon Relaxation Dynamics")
ax.grid()
ax.legend()